# XGBoost Gradient Boosted Trees
Implements a classic XGBoost-style gradient boosted tree model for predicting `blueWins`.

## Why XGBoost here?
- Boosting captures nonlinear thresholds (specific gold swings) that linear models could miss.
- Native handling of unscaled numeric features lowers preprocessing overhead.
- Feature importances and SHAP values come for free, keeping the modeling stack explainable.

### Hyperparameters worth tracking
| Hyperparameter | Impact | Typical sweep |
| --- | --- | --- |
| `max_depth` | Tree depth; shallower = more regularized. | 4 – 8 |
| `eta` (learning rate) | Shrinks each tree's contribution. | 0.02 – 0.1 |
| `subsample` / `colsample_bytree` | Row/feature subsampling per tree. | 0.6 – 0.9 |
| `min_child_weight` | Minimum Hessian in leaves; combats overfit. | 1 – 10 |
| `lambda` / `alpha` | L2/L1 regularization on leaf weights. | 0 – 2 |
| `num_boost_round` + `early_stopping_rounds` | Total trees vs. patience. | 400 – 1200 / 25 – 75 |

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / 'requirements.txt').exists():
        break
    REPO_ROOT = REPO_ROOT.parent
sys.path.append(str(REPO_ROOT / 'src'))

PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
TARGET = 'blueWins'
PROCESSED_DIR

PosixPath('/Users/liamsandy/ML_Project/data/processed')

## Load the canonical train/test split
These CSVs are produced by `src/preprocessing/run_eda.py` and keep comparisons fair across notebooks.

In [3]:
train_features = pd.read_csv(PROCESSED_DIR / 'train_features.csv')
test_features = pd.read_csv(PROCESSED_DIR / 'test_features.csv')
train_labels = pd.read_csv(PROCESSED_DIR / 'train_labels.csv')[TARGET]
test_labels = pd.read_csv(PROCESSED_DIR / 'test_labels.csv')[TARGET]

train_features.head()

,blueWardsPlaced,blueWardsDestroyed,blueFirstBlood,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,blueHeralds,blueTowersDestroyed,...,redGoldPerMin,blue_objective_share,blue_vision_share,blue_kill_share,log_blueTotalGold,log_blueTotalExperience,log_redTotalGold,log_redTotalExperience,log_blueGoldPerMin,log_redGoldPerMin
0,11,3,0,7,3,7,0,0,0,0,...,1499.0,0.0,0.440000,0.700000,9.708992,9.799848,9.615205,9.744961,7.406954,7.313220
1,91,3,1,6,1,4,2,1,1,0,...,1375.0,1.0,0.866667,0.857143,9.741968,9.864643,9.528867,9.792612,7.439912,7.226936
2,20,6,1,11,8,12,1,1,0,0,...,1690.4,1.0,0.571429,0.578947,9.870448,9.887053,9.735365,9.790711,7.568328,7.433312
3,16,2,1,2,11,2,1,1,0,0,...,1947.2,1.0,0.484848,0.153846,9.542015,9.658929,9.876784,9.870086,7.240076,7.574661
4,12,4,0,5,4,7,1,1,0,0,...,1488.5,0.5,0.400000,0.555556,9.710449,9.832689,9.608176,9.729491,7.408409,7.306196


## Create XGBoost DMatrices
Hold out 15% of the training split for early stopping, keep the official test split untouched for final metrics.

In [5]:
X_train, X_valid, y_train, y_valid = train_test_split(
    train_features,
    train_labels,
    test_size=0.15,
    stratify=train_labels,
    random_state=42,
)

dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)
dtest = xgb.DMatrix(test_features, label=test_labels)

X_train.shape, X_valid.shape, test_features.shape

((6717, 45), (1186, 45), (1976, 45))

## Train the booster
Configuration mimics common Kaggle defaults; tweak `eta`, `max_depth`, and regularization to explore the bias/variance trade-off.

In [7]:
params = {
    'objective': 'binary:logistic',
    'eval_metric': ['logloss', 'auc'],
    'eta': 0.05,
    'max_depth': 6,
    'subsample': 0.75,
    'colsample_bytree': 0.65,
    'lambda': 1.0,
    'alpha': 0.1,
    'min_child_weight': 5,
    'scale_pos_weight': 1.0,
    'tree_method': 'hist',
    'random_state': 42,
}

evals_result = {}
watchlist = [(dtrain, 'train'), (dvalid, 'valid')]
booster = xgb.train(
    params,
    dtrain,
    num_boost_round=1200,
    evals=watchlist,
    early_stopping_rounds=50,
    evals_result=evals_result,
    verbose_eval=50,
)

print(f"Best iteration: {booster.best_iteration}")

[0]	train-logloss:0.67850	train-auc:0.81627	valid-logloss:0.68003	valid-auc:0.78750
[50]	train-logloss:0.47322	train-auc:0.86762	valid-logloss:0.53297	valid-auc:0.81009
[57]	train-logloss:0.46651	train-auc:0.87006	valid-logloss:0.53175	valid-auc:0.80988
Best iteration: 7


## Evaluate on the untouched test split
Report standard metrics for parity with other baselines.

In [9]:
test_probs = booster.predict(dtest)
test_preds = (test_probs >= 0.5).astype(int)

print('Accuracy:', accuracy_score(test_labels, test_preds))
print('F1:', f1_score(test_labels, test_preds))
print('ROC-AUC:', roc_auc_score(test_labels, test_probs))

Accuracy: 0.7211538461538461
F1: 0.718158567774936
ROC-AUC: 0.8034815702665601


## Inspect feature importances
Use both gain-based importance and SHAP-style values for qualitative insight.

In [11]:
gain_importance = booster.get_score(importance_type='gain')
if gain_importance:
    top_gain = sorted(gain_importance.items(), key=lambda kv: kv[1], reverse=True)[:10]
    print('Top features by gain:')
    for feature, score in top_gain:
        print(f"{feature}: {score:.4f}")
else:
    print('No gain importances found (unexpected).')

Top features by gain:
blueGoldDiff: 46.4206
blueExperienceDiff: 28.8844
blue_kill_share: 28.4519
blueGoldPerMin: 14.2085
redTotalGold: 9.5913
blueDragons: 8.4323
blueTotalGold: 7.8197
redHeralds: 7.2716
redDragons: 6.6073
redTotalExperience: 6.2594


## Engineered feature experiment
Train on the engineered ratios/differences saved under `engineered_train.csv` to see whether compact representations help.

In [13]:
engineered_train = pd.read_csv(PROCESSED_DIR / 'engineered_train.csv')
engineered_test = pd.read_csv(PROCESSED_DIR / 'engineered_test.csv')

X_eng = engineered_train.drop(columns=[TARGET])
y_eng = engineered_train[TARGET]
X_eng_train, X_eng_valid, y_eng_train, y_eng_valid = train_test_split(
    X_eng,
    y_eng,
    test_size=0.15,
    stratify=y_eng,
    random_state=42,
)

deng_train = xgb.DMatrix(X_eng_train, label=y_eng_train)
deng_valid = xgb.DMatrix(X_eng_valid, label=y_eng_valid)
deng_test = xgb.DMatrix(engineered_test.drop(columns=[TARGET]), label=engineered_test[TARGET])

eng_params = {
    **params,
    'eta': 0.04,
    'max_depth': 5,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
}

eng_booster = xgb.train(
    eng_params,
    deng_train,
    num_boost_round=1000,
    evals=[(deng_train, 'train'), (deng_valid, 'valid')],
    early_stopping_rounds=40,
    verbose_eval=100,
)

eng_probs = eng_booster.predict(deng_test)
print('Engineered ROC-AUC:', roc_auc_score(engineered_test[TARGET], eng_probs))

[0]	train-logloss:0.68165	train-auc:0.81248	valid-logloss:0.68237	valid-auc:0.80395
[79]	train-logloss:0.48205	train-auc:0.85394	valid-logloss:0.52632	valid-auc:0.81501
Engineered ROC-AUC: 0.8031624562050526


### Takeaways
- Boosted trees usually squeeze out 1–3 ROC-AUC points over the TF linear baseline, especially by learning sharp thresholds on gold/XP.
- Feature importances consistently highlight gold/XP differentials and early objective control, aligning with domain expectations.
- Engineered ratios perform on par with raw stats while shrinking the feature set, making SHAP explanations easier to digest.